Step 0 — Pick a genuinely messy dataset. This challenge falls flat on clean data. Good candidates: the Ames Housing dataset (80 columns, plenty of missing values with different meanings), the Telco Customer Churn dataset (a numeric column secretly stored as text), or any real export from a CRM or logs if you have one. Define a prediction target up front — leakage hunting only makes sense relative to a target.

Step 1 — Profile before touching anything. Spend the first session only looking, never fixing. A quick pass:

In [ ]:
df.info()                      # dtypes and non-null counts
df.describe(include="all")     # ranges, uniques, top values
df.isna().mean().sort_values() # missing rate per column
df.duplicated().sum()

Then go one level deeper, because the interesting problems hide from describe(): check for impossible values (negative ages, dates in the future), columns where 95% of rows share one value, numeric columns with dtype object (usually a stray " " or "N/A" string), and categorical columns where "NYC", "nyc", and "New York" are three different values. Write your findings down as a short "case file" — one line per problem, with a proposed fix. That document is the real deliverable of this step.

Step 2 — Handle missing values with intent, not reflexes. The core lesson: why a value is missing determines what you do. A missing garage_year_built usually means "no garage" — fill it with a constant and add a has_garage flag. A missing income in survey data may mean "declined to answer," which is itself informative — an indicator column captures that. Only truly random gaps deserve a plain median/mode fill. Rule of thumb: whenever you impute, consider adding a was_missing boolean column alongside it; models often learn more from the flag than the filled value. And sometimes the right answer is dropping a column entirely — 80% missing with no pattern is noise, not signal.

Step 3 — Hunt for leakage. Leakage means information from the future or from the target sneaking into your features. Three checks catch most of it. First, correlation with the target: any single feature with suspiciously strong predictive power deserves interrogation ("days_in_hospital predicts hospitalization perfectly — because it's only recorded afterwards"). Second, the timeline test: for every column, ask "would I know this value at the moment I need to make the prediction?" Third, the procedural kind: fitting your imputer or scaler on the full dataset before splitting leaks test-set statistics into training. Split first, fit transforms on train only. To make it stick, deliberately build a leaky model, watch it score 99%, then remove the leak and watch reality return — nothing teaches this better.

Stretch — turn the cleanup into a Pipeline. The rewrite: every fix from steps 2–3 becomes a transformer instead of an ad-hoc cell. The backbone is ColumnTransformer routing numeric columns into one Pipeline(SimpleImputer(add_indicator=True), StandardScaler()) and categoricals into another with mode-imputation plus OneHotEncoder(handle_unknown="ignore"). For custom fixes like unifying "NYC" variants, write a small class with fit and transform methods (inherit from BaseEstimator and TransformerMixin). The payoff moment is calling cross_val_score on the whole pipeline: because imputation now happens inside each fold, the procedural leakage from step 3 becomes structurally impossible. That pipeline object is exactly what you'll serialize and serve in the Tier 4 challenges, so effort here pays off twice.